**1. MLOps - Skapa en organiserad mapp/fil struktur**

In [ ]:
import os

os.makedirs('/content/AI 1/assignment2/modeller', exist_ok=True)

with open('/content/AI 1/assignment2/modeller/info.txt', 'w') as f:
    f.write('Träningsinformation\n')
    f.write('Antal epochs: 5\n')
    f.write('Batch size: 64\n')
    f.write('Modell: Perceptron\n')

print("Mappar skapade!")


Mappar skapade!


Förberedelse inför träningen
1. Importerar alla verktyg vi behöver (PyTorch)
2. Laddar ner MNIST bilderna och bygger nätverket
3. Bygger nätverket (perceptorn)

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

train_data = torchvision.datasets.MNIST(
    root='./data', train=True,
    transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.MNIST(
    root='./data', train=False,
    transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=64)

class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10))
    def forward(self, x):
        x = x.view(-1, 784)
        return self.layers(x)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 478kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.0MB/s]


Spara modellen under träning för att se vad de har lärt sig:
1. tittar på en bild
2. Gissar vilken siffra det är
3. Kollar hur fel gissningen var
4. Justerar vikterna
5. Sparar modellen - 5 filer sparades på google drive

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Använder: {device}")

model = Perceptron().to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

    # Spara modellen efter varje epoch
    torch.save(model.state_dict(),
    f'/content/AI 1/assignment2/modeller/modell_epoch_{epoch+1}.pth')
    print(f"Epoch {epoch+1} klar! Modell sparad!")

print("Träning klar!")

Använder: cuda
Epoch 1 klar! Modell sparad!
Epoch 2 klar! Modell sparad!
Epoch 3 klar! Modell sparad!
Epoch 4 klar! Modell sparad!
Epoch 5 klar! Modell sparad!
Träning klar!


**2**.  **performance** **metrics**
1. Mäta hur långt träningen tar
2. Hur bra modellen presterar ( Hur många siffor den gissar rätt)

> Lägg till blockcitat



In [ ]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Använder: {device}")

model = Perceptron().to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# Mät total tid för hela träningen
total_start = time.time()

for epoch in range(5):
    # Mät tid per epoch
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

    # Mät noggrannhet efter varje epoch
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start

    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    # Spara modellen efter varje epoch
    torch.save(model.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Använder: cuda
Epoch 1 → Noggrannhet: 94.38% | Tid: 9.5s
Epoch 2 → Noggrannhet: 95.79% | Tid: 8.9s
Epoch 3 → Noggrannhet: 96.54% | Tid: 8.9s
Epoch 4 → Noggrannhet: 96.76% | Tid: 8.9s
Epoch 5 → Noggrannhet: 96.30% | Tid: 8.9s

Träning klar! Total tid: 45.1s


**3.** **Data augmentation**
Varierar bilderna konstgjort så att nätverket blir bättre på att känna igen siffror de aldrig sett innan.
Vi roterar, zoomar och lägger till brus på bilderna.

In [ ]:
import torchvision.transforms as transforms

# Ny träningsdata med augmentation
train_data_augmented = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transforms.Compose([
        transforms.RandomRotation(10),        # rotera upp till 10 grader
        transforms.RandomAffine(0, scale=(0.9, 1.1)),  # zooma lite
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))  # normalisera pixelvärdena
    ]),
    download=True
)

train_loader_augmented = torch.utils.data.DataLoader(
    train_data_augmented, batch_size=64)

print("Data augmentation klar!")
print("Bilderna roteras och zoomas slumpmässigt under träningen!")


Data augmentation klar!
Bilderna roteras och zoomas slumpmässigt under träningen!


In [ ]:
model_aug = Perceptron().to(device)
optimizer_aug = torch.optim.Adam(model_aug.parameters())
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(5):
    epoch_start = time.time()

    for images, labels in train_loader_augmented:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_aug.zero_grad()
        output = model_aug(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_aug.step()

    # Mät noggrannhet
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_aug(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    # Spara modellen
    torch.save(model_aug.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_aug_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Epoch 1 → Noggrannhet: 90.22% | Tid: 24.7s
Epoch 2 → Noggrannhet: 94.26% | Tid: 25.0s
Epoch 3 → Noggrannhet: 95.54% | Tid: 24.9s
Epoch 4 → Noggrannhet: 95.37% | Tid: 24.8s
Epoch 5 → Noggrannhet: 96.17% | Tid: 24.6s

Träning klar! Total tid: 124.0s


**4. CNN** - **Convolutional neural network ** **bold text**
Istället för vanliga lage\r så använder vi convolutional lager som är mycket bättre på att förstp bilder.
CNN ser möster, kanter och former i bilden istället för bara en lång lista av pixlar som ANN gör.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Första convolutional lagret
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Andra convolutional lagret
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(-1, 64 * 7 * 7)
        return self.fc_layers(x)

print("CNN modell skapad!")

CNN modell skapad!


Träning med CNN

In [ ]:

model_cnn = CNN().to(device)
optimizer_cnn = torch.optim.Adam(model_cnn.parameters())
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(5):
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_cnn.zero_grad()
        output = model_cnn(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_cnn.step()

    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_cnn(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    torch.save(model_cnn.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_cnn_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Epoch 1 → Noggrannhet: 97.43% | Tid: 10.4s
Epoch 2 → Noggrannhet: 98.79% | Tid: 9.8s
Epoch 3 → Noggrannhet: 98.86% | Tid: 9.4s
Epoch 4 → Noggrannhet: 98.75% | Tid: 9.4s
Epoch 5 → Noggrannhet: 98.70% | Tid: 9.6s

Träning klar! Total tid: 48.6s


5.Arkitektur - Hur ett nätverk är uppbyggd, hur många lager och vilka typer av lager bold text

Vi testar en enklare arkitektur med bara ett convolutional lager
och jämför resultatet med vår tidigare CNN som hade två lager.
Färre lager = enklare modell, men är den lika bra?

In [ ]:
class CNN_1lager(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Bara ETT convolutional lager
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(32 * 14 * 14, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(-1, 32 * 14 * 14)
        return self.fc_layers(x)

print("CNN med ett lager skapad!")

CNN med ett lager skapad!


Träning med CNN ett lager
Tränar vår enklare CNN och mäter noggrannheten
så vi kan jämföra med vår tidigare CNN med två lager!


In [ ]:
model_cnn1 = CNN_1lager().to(device)
optimizer_cnn1 = torch.optim.Adam(model_cnn1.parameters())
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(5):
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_cnn1.zero_grad()
        output = model_cnn1(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_cnn1.step()

    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_cnn1(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    torch.save(model_cnn1.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_cnn1_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Epoch 1 → Noggrannhet: 96.98% | Tid: 8.6s
Epoch 2 → Noggrannhet: 98.19% | Tid: 9.3s
Epoch 3 → Noggrannhet: 98.53% | Tid: 9.2s
Epoch 4 → Noggrannhet: 98.46% | Tid: 9.3s
Epoch 5 → Noggrannhet: 98.51% | Tid: 8.5s

Träning klar! Total tid: 44.9s


CNN med tre convolutional lager
Vi testar en mer komplex arkitektur med tre convolutional lager
och jämför resultatet med ett och två lager.
Fler lager = mer komplex modell, men är den bättre?

In [ ]:
class CNN_3lager(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Första lagret
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Andra lagret
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Tredje lagret
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(-1, 128 * 7 * 7)
        return self.fc_layers(x)

print("CNN med tre lager skapad!")

CNN med tre lager skapad!


In [ ]:
model_cnn3 = CNN_3lager().to(device)
optimizer_cnn3 = torch.optim.Adam(model_cnn3.parameters())
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(5):
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_cnn3.zero_grad()
        output = model_cnn3(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_cnn3.step()

    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_cnn3(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    torch.save(model_cnn3.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_cnn3_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Epoch 1 → Noggrannhet: 97.85% | Tid: 10.1s
Epoch 2 → Noggrannhet: 98.79% | Tid: 10.4s
Epoch 3 → Noggrannhet: 98.78% | Tid: 10.1s
Epoch 4 → Noggrannhet: 98.80% | Tid: 9.5s
Epoch 5 → Noggrannhet: 99.11% | Tid: 9.8s

Träning klar! Total tid: 49.9s


5.Regularization -  förhindra overfitting
Vi lägger till dropout och batch normalization i vårt bästa nätverk
(CNN med 3 lager) för att förhindra overfitting och göra
nätverket bättre på bilder det aldrig sett innan!


In [ ]:
class CNN_Regularized(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Första lagret med batch normalization
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),    # normaliserar efter conv lagret
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),      # stänger av 25% av neuronerna

            # Andra lagret med batch normalization
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            # Tredje lagret med batch normalization
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(0.25)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 7 * 7, 128),
            nn.BatchNorm1d(128),   # normaliserar i fc lagret också
            nn.ReLU(),
            nn.Dropout(0.5),       # stänger av 50% innan sista lagret
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(-1, 128 * 7 * 7)
        return self.fc_layers(x)

print("CNN med regularization skapad!")

CNN med regularization skapad!


In [ ]:
model_reg = CNN_Regularized().to(device)
optimizer_reg = torch.optim.Adam(
    model_reg.parameters(),
    weight_decay=1e-4    # weight decay här!
)
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(5):
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_reg.zero_grad()
        output = model_reg(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_reg.step()

    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_reg(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    torch.save(model_reg.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_reg_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Epoch 1 → Noggrannhet: 97.85% | Tid: 10.9s
Epoch 2 → Noggrannhet: 98.27% | Tid: 10.3s
Epoch 3 → Noggrannhet: 98.33% | Tid: 10.9s
Epoch 4 → Noggrannhet: 98.35% | Tid: 10.7s
Epoch 5 → Noggrannhet: 98.69% | Tid: 10.6s

Träning klar! Total tid: 53.4s


7. Hyperparameter tuning
Vi skapar en lista med olika kombinationer av inställningar
och låter datorn testa alla kombinationer automatiskt!
Sedan jämför vi resultaten för att hitta den bästa kombinationen.

In [ ]:

import itertools

# Lista med olika hyperparametrar att testa
hyperparams = [
    {
        'learning_rate': 0.001,
        'dropout': 0.25,
        'filters': 32,
        'kernel_size': 3
    },
    {
        'learning_rate': 0.0001,
        'dropout': 0.5,
        'filters': 64,
        'kernel_size': 3
    },
    {
        'learning_rate': 0.001,
        'dropout': 0.3,
        'filters': 32,
        'kernel_size': 5
    },
    {
        'learning_rate': 0.0005,
        'dropout': 0.4,
        'filters': 64,
        'kernel_size': 5
    },
    {
        'learning_rate': 0.001,
        'dropout': 0.25,
        'filters': 128,
        'kernel_size': 3
    }
]

print(f"Antal kombinationer att testa: {len(hyperparams)}")
print("Klar!")


Antal kombinationer att testa: 5
Klar!


In [ ]:

class CNN_Tunable(nn.Module):
    def __init__(self, filters, kernel_size, dropout):
        super().__init__()
        self.filters = filters
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, filters, kernel_size=kernel_size, padding=1),
            nn.BatchNorm2d(filters),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout),

            nn.Conv2d(filters, filters*2, kernel_size=kernel_size, padding=1),
            nn.BatchNorm2d(filters*2),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout)
        )

        # Räkna ut rätt storlek automatiskt!
        self._feature_size = self._get_feature_size()

        self.fc_layers = nn.Sequential(
            nn.Linear(self._feature_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10)
        )

    def _get_feature_size(self):
        # Skickar en testbild genom conv lagren för att
        # räkna ut rätt storlek automatiskt!
        test = torch.zeros(1, 1, 28, 28).to(next(self.parameters()).device) if len(list(self.parameters())) > 0 else torch.zeros(1, 1, 28, 28)
        out = self.conv_layers(test)
        return out.view(1, -1).size(1)

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # räknar ut storleken automatiskt
        return self.fc_layers(x)

print("Tunable CNN fixad!")

Tunable CNN fixad!


In [ ]:
resultat = []

for i, params in enumerate(hyperparams):
    print(f"\nKörning {i+1}/{len(hyperparams)}")
    print(f"Parametrar: {params}")

    # Skapa modell med dessa hyperparametrar
    model_tuned = CNN_Tunable(
        filters=params['filters'],
        kernel_size=params['kernel_size'],
        dropout=params['dropout']
    ).to(device)

    optimizer_tuned = torch.optim.Adam(
        model_tuned.parameters(),
        lr=params['learning_rate']
    )
    criterion = nn.CrossEntropyLoss()

    # Träna i 3 epochs per kombination
    for epoch in range(3):
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer_tuned.zero_grad()
            output = model_tuned(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer_tuned.step()

    # Mät noggrannhet
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_tuned(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total

    # Spara resultatet
    resultat.append({
        'körning': i+1,
        'params': params,
        'noggrannhet': noggrannhet
    })

    print(f"Noggrannhet: {noggrannhet:.2f}%")

    # Spara modellen
    torch.save(model_tuned.state_dict(),
        f'/content/AI 1/assignment2/modeller/modell_tuned_{i+1}.pth')

print("\nHyperparameter tuning klar!")


Körning 1/5
Parametrar: {'learning_rate': 0.001, 'dropout': 0.25, 'filters': 32, 'kernel_size': 3}
Noggrannhet: 98.24%

Körning 2/5
Parametrar: {'learning_rate': 0.0001, 'dropout': 0.5, 'filters': 64, 'kernel_size': 3}
Noggrannhet: 95.91%

Körning 3/5
Parametrar: {'learning_rate': 0.001, 'dropout': 0.3, 'filters': 32, 'kernel_size': 5}
Noggrannhet: 98.12%

Körning 4/5
Parametrar: {'learning_rate': 0.0005, 'dropout': 0.4, 'filters': 64, 'kernel_size': 5}
Noggrannhet: 98.21%

Körning 5/5
Parametrar: {'learning_rate': 0.001, 'dropout': 0.25, 'filters': 128, 'kernel_size': 3}
Noggrannhet: 97.87%

Hyperparameter tuning klar!


In [ ]:

print("=== Resultat hyperparameter tuning ===\n")

# Sortera efter noggrannhet
resultat_sorted = sorted(resultat, key=lambda x: x['noggrannhet'], reverse=True)

for r in resultat_sorted:
    print(f"Körning {r['körning']} → Noggrannhet: {r['noggrannhet']:.2f}%")
    print(f"  Learning rate: {r['params']['learning_rate']}")
    print(f"  Dropout:       {r['params']['dropout']}")
    print(f"  Filter:        {r['params']['filters']}")
    print(f"  Kernel size:   {r['params']['kernel_size']}")
    print()

bäst = resultat_sorted[0]
print(f"Bästa kombinationen: Körning {bäst['körning']} med {bäst['noggrannhet']:.2f}%")

=== Resultat hyperparameter tuning ===

Körning 1 → Noggrannhet: 98.24%
  Learning rate: 0.001
  Dropout:       0.25
  Filter:        32
  Kernel size:   3

Körning 4 → Noggrannhet: 98.21%
  Learning rate: 0.0005
  Dropout:       0.4
  Filter:        64
  Kernel size:   5

Körning 3 → Noggrannhet: 98.12%
  Learning rate: 0.001
  Dropout:       0.3
  Filter:        32
  Kernel size:   5

Körning 5 → Noggrannhet: 97.87%
  Learning rate: 0.001
  Dropout:       0.25
  Filter:        128
  Kernel size:   3

Körning 2 → Noggrannhet: 95.91%
  Learning rate: 0.0001
  Dropout:       0.5
  Filter:        64
  Kernel size:   3

Bästa kombinationen: Körning 1 med 98.24%
